# SMS Pipeline — Reference Notebook

Cell-by-cell walkthrough of the **live** SMS pipeline: `ml_service/app/services/sms_pipeline.py` (the five stages) and `ml_service/app/parsers/sms_parser.py` (classification + extraction).

**This notebook imports and runs the real pipeline code — it does not reimplement any logic.** That is deliberate: the `.py` modules are the single source of truth (used by both the live FastAPI ingest route and this offline batch run), and duplicating regex/logic here would drift out of sync the first time either changes. Several cells below print the actual source of key functions (`inspect.getsource`) so you can read the logic without switching files, but the *execution* always calls the real code.

See `docs/sms_pipeline.md` for the full architecture writeup.

**Data note:** this reads `ml_service/data/captured_sms.csv`, which is real personal financial data and is gitignored — never commit its contents.

In [ ]:
import sys
import os
import inspect

import pandas as pd

# ml_service/ isn't a package on sys.path by default — add it so `from app...` imports work
# regardless of where this notebook is launched from.
ML_SERVICE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', 'ml_service'))
if ML_SERVICE_ROOT not in sys.path:
    sys.path.insert(0, ML_SERVICE_ROOT)

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)

from app.services.sms_pipeline import SmsPipeline
import app.parsers.sms_parser as sms_parser

pipeline = SmsPipeline()
print('Reading from:', pipeline.input_file)

## Stage 0 — Load raw capture

`data/captured_sms.csv` is written by `app/routes/ingest.py` and stores **only** what the phone actually sent — `id, sender, body, timestamp_ms, timestamp_human, device_id`. No parser output is baked in, so this file stays valid no matter how the parser below evolves.

In [ ]:
raw = pipeline.load()
print('shape:', raw.shape)
print('columns:', list(raw.columns))
raw.head()

## Stage 1 — Structural clean (`structural_clean`)

Content-agnostic, runs before any interpretation:
- drop exact resends (`body` + `timestamp_ms`)
- normalize mixed timestamp formats (epoch-ms or ISO string) → **IST** (`Asia/Kolkata`) — the user + banks are Indian, so dates are attributed to the correct local calendar day rather than UTC
- drop rows with an unparseable timestamp (counted, never silently dropped)
- sort oldest-first, derive a plain `date` column

**No year filter** — the raw file legitimately spans several years.

In [ ]:
print(inspect.getsource(pipeline.structural_clean))

In [ ]:
cleaned = pipeline.structural_clean(raw)
print('raw rows:       ', len(raw))
print('after cleaning: ', len(cleaned))
print('date range:     ', cleaned['event_time'].min(), '->', cleaned['event_time'].max())
cleaned['event_time'].dt.year.value_counts().sort_index()

## Stage 2 + 3 — Classify and extract (`classify_and_extract`)

One `parse_sms_body()` call per row does both classification (label + confidence) and, for financial rows, field extraction (amount, direction, bank, mode, recipient, UPI ID, ref ID, balance). One pass, not two — and this is the exact function the live FastAPI ingest route calls too.

A message only reaches `FINANCIAL_TRANSACTION` if it has an amount, a direction, and financial context; anything short of that is demoted to `UNKNOWN` for the review queue rather than guessed.

In [ ]:
print(inspect.getsource(sms_parser.parse_sms_body))

### Classification internals (`_classify_sms`)

Order matters here — OTP, then the scam gate, then collect-requests, then mandate setups, then payment-confirmation recovery, then the general promo gate, then the structured financial check. Each gate is there because of a specific false positive/negative found by auditing real data (see `docs/sms_pipeline.md` — "Scam gate" and "Payment-confirmation recovery").

In [ ]:
print(inspect.getsource(sms_parser._classify_sms))

### Key patterns referenced above

- `PHONE_SENDER_PATTERN` / `URL_SHORTENER_PATTERN` / `GAMBLING_PATTERN` — the scam gate. Fake-transaction spam ("Transaction done of Rs.98,650 to your Rummy A/C") mimics real wording; left unchecked it injected ~₹381k of phantom money into the financial set.
- `COLLECT_REQUEST_PATTERN` — "has requested Rs500... once approved" is a request to pay, not a completed transaction.
- `MANDATE_SETUP_PATTERN` — a UPI-Mandate/autopay *setup* SMS moves no money; the real recurring debit arrives later as its own SMS.
- `PAYMENT_CONFIRMATION_PATTERN` / `TELECOM_CONFIRMATION_PATTERN` — recovers real outgoing payments that don't use "debited" wording (telecom recharges, INB transfers, booking payments), forces them to DEBIT, and suppresses their per-SMS "Order Id" from being treated as a bank `ref_id` (it isn't one — keeping it would stop the 2-4 SMS of one recharge from deduping in Stage 4).

In [ ]:
for name in [
    'PHONE_SENDER_PATTERN', 'URL_SHORTENER_PATTERN', 'GAMBLING_PATTERN',
    'COLLECT_REQUEST_PATTERN', 'MANDATE_SETUP_PATTERN',
    'PAYMENT_CONFIRMATION_PATTERN', 'TELECOM_CONFIRMATION_PATTERN',
]:
    pattern = getattr(sms_parser, name)
    print(f'{name}:')
    print(' ', pattern.pattern)
    print()

### Amount extraction — the balance-confusion fix

Some SBI formats put the balance far from the amount ("Debited INR 10,000 … Avl Balance INR 42,916.45"); a naive scorer picked the larger balance. `_extract_transaction_amount` uses `BALANCE_PATTERN` to pinpoint the balance figure and excludes that exact span from amount candidates.

In [ ]:
print(inspect.getsource(sms_parser._extract_transaction_amount))

### Run classify + extract over the full cleaned dataset

In [ ]:
df = pipeline.classify_and_extract(cleaned)
df['classification_label'].value_counts()

In [ ]:
financial = df[df['classification_label'].eq('FINANCIAL_TRANSACTION')].copy()

def extraction_rate(col):
    return f'{100 * financial[col].notna().mean():.1f}%'

print('financial rows:', len(financial))
for col in ['amount', 'direction', 'bank', 'ref_id', 'recipient_name', 'upi_id']:
    print(f'  {col:16s} {extraction_rate(col)}')

## Stage 4 — Transaction dedup (`dedup_transactions`)

Two passes:

1. **ref_id (authoritative).** Two rows sharing a non-null `ref_id` are the same transaction no matter the time gap or telco gateway. Different ref_ids stay distinct — SBI assigns a fresh reference per payment.
2. **Time-window clusters with an identity rule.** Adjacent same-(amount, direction) rows within a window are clustered, then the cluster's transaction count is decided as `max(distinct target-phones, distinct ref_ids, 1)`. This is what lets one telecom recharge (bank debit + 2-3 operator confirmations, one phone) collapse to **one** row, while two family numbers recharged for the same amount minutes apart (two phones) correctly stay **two**.

Adjacency (not a fixed clock bucket) is deliberate — floor-bucketing split genuine twins that straddled a boundary (e.g. 19s apart across :50/:52).

In [ ]:
print(inspect.getsource(pipeline.dedup_transactions))

In [ ]:
financial_deduped = pipeline.dedup_transactions(financial)
print('financial before dedup:', len(financial))
print('financial after dedup: ', len(financial_deduped))
print('direction split:', financial_deduped['direction'].value_counts().to_dict())

## Review queue (`build_review_queue`)

Every `UNKNOWN` or sub-0.75-confidence row, ready for a manual labeling pass — the raw material for a future supervised classifier.

In [ ]:
review = pipeline.build_review_queue(df)
print('review queue rows:', len(review))
review.head()

## Stage 5 — Validate (`validate`)

Content-level checks, not just a rate table — in particular the `amount == balance` suspect count, which is what originally surfaced the balance-confusion bug above.

In [ ]:
import json
print(json.dumps(pipeline.validate(financial_deduped), indent=2, default=str))

## Cross-check against the bank statement

The SMS pipeline only dedupes SMS-against-SMS — it does **not** know about the independently-produced bank workbook. Many SMS transactions also exist in the statement (by `ref_id`, or by same date+amount+direction when the ref differs/is missing), so merging both sources without this check would double-count. This mirrors the manual audit already done in `data/manual_review_error_found.xlsx` / `tru_financial_deplicate.xlsx` / `ambiguous_matches_review.xlsx`.

In [ ]:
import re

def clean_ref(v):
    s = str(v).strip()
    if s.lower() in ('nan', 'none', ''):
        return None
    s = re.sub(r'\.0$', '', s)
    return re.sub(r'[^A-Za-z0-9]', '', s).upper() or None

WORKBOOK_PATH = r'C:\Users\yashs\Desktop\Journey\SpendWise\ml_preprocessing\CSVS\SpendWise_4yrs_Clean_Merchants.xlsx'

sms = financial_deduped.copy()
sms['ref_norm'] = sms['ref_id'].map(clean_ref)
sms['day'] = sms['event_time'].dt.tz_convert('Asia/Kolkata').dt.date

wb = pd.read_excel(WORKBOOK_PATH)
wb['ref_norm'] = wb['Transaction_ID'].map(clean_ref)
wb['day'] = pd.to_datetime(wb['Transaction_Date']).dt.date
wb['dir'] = wb['DR/CR_Indicator'].map({'DR': 'DEBIT', 'CR': 'CREDIT'})
wb['amt'] = wb['Amount'].abs()

wb_refs = set(wb['ref_norm'].dropna())
exact_ref_match = sms['ref_norm'].isin(wb_refs).sum()

sms_only = sms[~sms['ref_norm'].isin(wb_refs)]
found_by_date, ambiguous, genuinely_absent = 0, 0, 0
for _, r in sms_only.iterrows():
    cand = wb[(wb['day'] == r['day']) & (wb['amt'] == r['amount']) & (wb['dir'] == r['direction'])]
    if len(cand) == 1:
        found_by_date += 1
    elif len(cand) > 1:
        ambiguous += 1
    else:
        genuinely_absent += 1

print('SMS financial transactions:      ', len(sms))
print('  exact ref_id match in statement:', exact_ref_match)
print('  date+amount+direction match:    ', found_by_date)
print('  ambiguous (needs manual review):', ambiguous)
print('  genuinely SMS-only:             ', genuinely_absent)
print('  (sums to', exact_ref_match + found_by_date + ambiguous + genuinely_absent, ')')

## Save outputs

Same three files the CLI entrypoint (`python -m app.services.sms_pipeline`) writes. Run this cell only if you want to regenerate them from here instead.

In [ ]:
# summary = pipeline.run(write=True)
# print(json.dumps(summary, indent=2, default=str))